# P3: exploración de perfiles de uso por estación

Exploración para la pregunta de clustering: convertir los viajes de octubre de 2023 a septiembre de 2026 en perfiles comparables de salidas y llegadas por estación, día de semana y hora. Solo se leen las columnas necesarias para P3; no se cargan identificadores de viaje/usuario ni datos demográficos. La ubicación y capacidad se mantienen como metadatos para describir los grupos, no como variables de agrupamiento.

Se construyen perfiles para entrenamiento (octubre de 2023–junio de 2025), validación (julio–septiembre de 2025), prueba temporal (octubre de 2025–septiembre de 2026) y el periodo completo. Los perfiles de validación/prueba sirven para revisar estabilidad respecto a grupos ajustados en entrenamiento; el perfil completo respalda la descripción final del periodo acordado.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import pyarrow.parquet as pq

PROJ_ROOT = Path.cwd().resolve()
while not (PROJ_ROOT / "data" / "processed").is_dir() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
sys.path.insert(0, str(PROJ_ROOT))

from proyecto_mibici.features_p3 import (
    PROFILE_COLUMNS,
    aggregate_p3_events,
    attach_p3_station_metadata,
    build_p3_station_profiles,
)

PARQUET_PATH = PROJ_ROOT / "data" / "processed" / "raw_data.parquet"
STATIONS_PATH = PROJ_ROOT / "data" / "raw" / "nomenclatura_2026_09.csv"
PERIODS = {
    "train": (pd.Timestamp("2023-10-01"), pd.Timestamp("2025-07-01")),
    "validation": (pd.Timestamp("2025-07-01"), pd.Timestamp("2025-10-01")),
    "test": (pd.Timestamp("2025-10-01"), pd.Timestamp("2026-10-01")),
    "full": (pd.Timestamp("2023-10-01"), pd.Timestamp("2026-10-01")),
}

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(f"No se encontró el Parquet: {PARQUET_PATH}")
if not STATIONS_PATH.is_file():
    raise FileNotFoundError(f"No se encontró la nomenclatura: {STATIONS_PATH}")

2026-10-18 17:25:55.928 | INFO     | proyecto_mibici.config:<module>:11 - PROJ_ROOT path is: /Users/beibibumer/Documents/GitHub/TC5061-Proyecto-Equipo25


## Agregar eventos de salida y llegada

Cada viaje válido aporta una salida en `Inicio_del_viaje` para `Origen_Id` y una llegada en `Fin_del_viaje` para `Destino_Id`. Los eventos se asignan a periodos usando la fecha-hora del evento y se agregan por estación, dirección, mes, día de semana y hora. Se excluyen filas con fecha inválida, estación vacía o fin anterior al inicio; el conteo de exclusiones queda visible para la auditoría.

In [2]:
required_columns = [
    "Inicio_del_viaje",
    "Fin_del_viaje",
    "Origen_Id",
    "Destino_Id",
    "mes_archivo",
]
audit = {}
event_chunks = []
parquet_file = pq.ParquetFile(PARQUET_PATH)

for batch in parquet_file.iter_batches(batch_size=250_000, columns=required_columns):
    trips = batch.to_pandas()
    trips = trips.loc[trips["mes_archivo"].between("2023-10", "2026-09")]
    if trips.empty:
        continue

    counts, batch_audit = aggregate_p3_events(
        trips,
        start=PERIODS["full"][0],
        end=PERIODS["full"][1],
    )
    for audit_name, audit_value in batch_audit.items():
        audit[audit_name] = audit.get(audit_name, 0) + int(audit_value)
    if not counts.empty:
        event_chunks.append(counts)

event_counts = (
    pd.concat(event_chunks, ignore_index=True)
    .groupby(
        ["estacion", "mes", "direccion", "dia_semana", "hora"],
        observed=True,
        as_index=False,
    )["viajes"]
    .sum()
)
audit_summary = pd.Series(audit, name="registros").to_frame()
audit_summary

,registros
viajes_entrada,13621885
fechas_inicio_invalidas,0
fechas_fin_invalidas,0
ids_estacion_vacios,0
fin_antes_de_inicio,0
viajes_validos,13621885
salidas_fuera_de_ventana,0
llegadas_fuera_de_ventana,17


## Construir una fila de features por estación

El perfil contiene 336 variables: salidas y llegadas para cada una de las 168 combinaciones de día de semana × hora. Primero se calcula la tasa promedio por ocurrencia de cada franja en el calendario del periodo; luego se normaliza el vector por estación para comparar la forma temporal y la mezcla salida/llegada, sin que el volumen bruto o que algunos días de semana aparezcan más veces domine la similitud. El volumen total y los meses con actividad se conservan como descriptores para interpretar/calibrar cobertura, pero no entran al agrupamiento. No se imputan perfiles de estaciones sin actividad en un periodo.

In [3]:
profiles = {}
descriptors = {}
for period_name in PERIODS:
    profiles[period_name], descriptors[period_name] = build_p3_station_profiles(
        event_counts,
        start=PERIODS[period_name][0],
        end=PERIODS[period_name][1],
    )
    if profiles[period_name].shape[1] != len(PROFILE_COLUMNS):
        raise ValueError(f"Número inesperado de features en {period_name}")
    if not profiles[period_name].sum(axis=1).between(0.999999, 1.000001).all():
        raise ValueError(f"Hay perfiles no normalizados en {period_name}")

catalog = pd.read_csv(STATIONS_PATH, dtype={"id": "string"})
full_station_ids = set(profiles["full"].index)
catalog_ids = set(catalog["id"].astype("string").str.strip())
monthly_coverage = descriptors["full"]["meses_con_actividad"]
stations_with_sparse_coverage = int(monthly_coverage.le(12).sum())
catalog_missing_coordinates = int(
    catalog[["latitude", "longitude"]].isna().any(axis=1).sum()
)
station_table = attach_p3_station_metadata(
    profiles["full"], descriptors["full"], catalog
)
profile_summary = pd.DataFrame(
    {
        "estaciones_activas": [profiles[name].shape[0] for name in PERIODS],
        "features_por_estacion": [profiles[name].shape[1] for name in PERIODS],
    },
    index=pd.Index(PERIODS, name="periodo"),
)

print(f"Viajes válidos dentro de los 36 CSV: {audit['viajes_validos']:,}")
print(f"Eventos de salida/llegada agregados en los 36 meses: {descriptors['full']['eventos_totales'].sum():,}")
print(f"Salidas/llegadas fuera de la ventana: {audit['salidas_fuera_de_ventana']:,}/{audit['llegadas_fuera_de_ventana']:,}")
print(f"IDs de estación observados en el periodo: {len(full_station_ids)}")
print(f"IDs observados sin coincidencia en nomenclatura: {len(full_station_ids - catalog_ids)}")
print(f"IDs de nomenclatura sin viajes observados: {len(catalog_ids - full_station_ids)}")
print(f"Estaciones con actividad en 12 meses o menos: {stations_with_sparse_coverage}")
print(f"Filas de catálogo con coordenadas faltantes: {catalog_missing_coordinates}")
print(f"Estaciones activas en entrenamiento, validación y prueba: {len(set.intersection(*(set(profiles[name].index) for name in ['train', 'validation', 'test'])))}")
display(profile_summary)
display(monthly_coverage.describe().to_frame())

Viajes válidos dentro de los 36 CSV: 13,621,885
Eventos de salida/llegada agregados en los 36 meses: 27,243,753
Salidas/llegadas fuera de la ventana: 0/17
IDs de estación observados en el periodo: 473
IDs observados sin coincidencia en nomenclatura: 2
IDs de nomenclatura sin viajes observados: 13
Estaciones con actividad en 12 meses o menos: 104
Filas de catálogo con coordenadas faltantes: 0
Estaciones activas en entrenamiento, validación y prueba: 367


,estaciones_activas,features_por_estacion
periodo,,
train,371,336
validation,367,336
test,471,336
full,473,336


,meses_con_actividad
count,473.000000
mean,28.403805
std,13.313517
min,1.000000
25%,33.000000
50%,36.000000
75%,36.000000
max,36.000000


## Lectura para el diseño de P3

- El volumen bruto no es una buena feature de similitud si se busca agrupar por forma de uso; queda como descriptor externo y se comparan proporciones por estación.
- Las coordenadas sirven para mapear e interpretar los grupos, no para ajustar los grupos de comportamiento. Así, la distancia geográfica no fuerza similitud de uso.
- La nomenclatura de septiembre de 2026 es una instantánea: 2 IDs observados no aparecen en ella y 13 IDs de catálogo no tienen viajes observados en los 36 meses. Los primeros pueden conservar perfil, pero no ubicación validada; los segundos no reciben un perfil artificial de ceros.
- La cobertura temporal es desigual. Las métricas de estabilidad entre entrenamiento/validación/prueba se calcularán solo sobre estaciones con actividad suficiente en los periodos comparados; los IDs nuevos o intermitentes se reportarán aparte.
- La nomenclatura actual no prueba que las coordenadas/capacidades correspondan a toda la historia. Capacidad queda fuera de las features; la ubicación se adjunta después del clustering y se marca su limitación temporal.

Este notebook valida la preparación de perfiles; la selección final del algoritmo, del número de grupos y de las métricas se cerrará en el apartado de modelos y métricas.